# Retention Cohort Analysis - Dorje Teas

[SYNTHETIC DATA - Illustrative only. No internal Dorje data used.]

Purpose: analyze repeat purchase, cohort retention, subscription attach, and LTV:CAC signals for a premium Darjeeling tea D2C model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

DATA_DIR = Path('../04_data_model')
if not DATA_DIR.exists():
    DATA_DIR = Path('04_data_model')
assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR.resolve()}'

def rupee(value):
    return f'Rs. {value:,.0f}'

cohorts = pd.read_csv(DATA_DIR / 'sample_customer_cohorts.csv')
orders = pd.read_csv(DATA_DIR / 'sample_orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'], dayfirst=True)
cohorts.head()

## Load and Validate Cohort Data

In [ ]:
required = {'cohort_month','new_customers','m1_retention_rate_pct','m2_retention_rate_pct','m3_retention_rate_pct','m4_retention_rate_pct','m5_retention_rate_pct','m6_retention_rate_pct','cumulative_ltv_m6','cac_cohort','ltv_cac_ratio_m6','subscription_attach_m2_pct','subscription_monthly_rev'}
missing = required - set(cohorts.columns)
assert not missing, missing
for _, row in cohorts.iterrows():
    vals = [row[f'm{i}_retention_rate_pct'] for i in range(1,7)]
    assert all(vals[i] >= vals[i+1] for i in range(len(vals)-1)), (row['cohort_month'], vals)
validation = cohorts[['cohort_month','new_customers','m1_retention_rate_pct','m3_retention_rate_pct','ltv_cac_ratio_m6']]
validation

## M0 to M6 Retention Heatmap

In [ ]:
retention_cols = [f'm{i}_retention_rate_pct' for i in range(1,7)]
retention_matrix = cohorts.set_index('cohort_month')[retention_cols]
fig, ax = plt.subplots(figsize=(14,8))
sns.heatmap(retention_matrix, annot=True, fmt='.1f', cmap='YlGnBu', linewidths=0.5, ax=ax)
ax.set_title('Dorje Teas Cohort Retention Heatmap [SYNTHETIC DATA]')
ax.set_xlabel('Months after first purchase')
ax.set_ylabel('Cohort month')
plt.tight_layout()
plt.show()

## Retention Curves for Selected Cohorts

In [ ]:
selected = cohorts[cohorts['cohort_month'].isin(['2024-04','2024-10','2025-04'])]
curve = selected.melt(id_vars='cohort_month', value_vars=retention_cols, var_name='month_number', value_name='retention_pct')
curve['month_number'] = curve['month_number'].str.extract('(\d)').astype(int)
ax = sns.lineplot(data=curve, x='month_number', y='retention_pct', hue='cohort_month', marker='o')
ax.set_title('Dorje Teas Selected Cohort Retention Curves [SYNTHETIC DATA]')
ax.set_xlabel('Months after first purchase')
ax.set_ylabel('Retention rate (%)')
plt.tight_layout()
plt.show()

## Average Retention Curve

In [ ]:
avg_retention = retention_matrix.mean().reset_index()
avg_retention.columns = ['month_number','retention_pct']
avg_retention['month_number'] = avg_retention['month_number'].str.extract('(\d)').astype(int)
ax = sns.lineplot(data=avg_retention, x='month_number', y='retention_pct', marker='o', color='#4C78A8')
ax.set_title('Dorje Teas Average Retention Curve [SYNTHETIC DATA]')
ax.set_xlabel('Months after first purchase')
ax.set_ylabel('Average retention rate (%)')
for _, row in avg_retention.iterrows():
    ax.text(row['month_number'], row['retention_pct'], f"{row['retention_pct']:.1f}%", ha='center', va='bottom')
plt.tight_layout()
plt.show()

## Cumulative LTV and LTV:CAC by Cohort

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,5))
sns.barplot(data=cohorts, x='cohort_month', y='cumulative_ltv_m6', ax=axes[0], color='#4C78A8')
sns.barplot(data=cohorts, x='cohort_month', y='ltv_cac_ratio_m6', ax=axes[1], color='#54A24B')
axes[0].set_title('Cumulative LTV by Cohort')
axes[1].set_title('LTV:CAC by Cohort')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Cohort month')
axes[0].set_ylabel('LTV (Rs.)')
axes[1].set_ylabel('LTV:CAC ratio')
fig.suptitle('Dorje Teas Cohort Value Diagnostics [SYNTHETIC DATA]')
plt.tight_layout()
plt.show()
cohorts[['cohort_month','new_customers','cumulative_ltv_m6','cac_cohort','ltv_cac_ratio_m6']]

## Time to Second Purchase from Orders Data

In [ ]:
customer_orders = orders.sort_values(['customer_id','order_date']).groupby('customer_id')['order_date'].agg(list).reset_index()
customer_orders['order_count'] = customer_orders['order_date'].str.len()
customer_orders['time_to_second_purchase_days'] = customer_orders['order_date'].apply(lambda dates: (dates[1] - dates[0]).days if len(dates) > 1 else np.nan)
time_to_second = customer_orders.dropna(subset=['time_to_second_purchase_days'])
summary_t2 = time_to_second['time_to_second_purchase_days'].describe().to_frame('days')
ax = sns.histplot(time_to_second['time_to_second_purchase_days'], bins=20, color='#F58518')
ax.axvline(21, color='black', linestyle='--', label='Day 21')
ax.axvline(30, color='gray', linestyle='--', label='Day 30')
ax.axvline(45, color='red', linestyle='--', label='Day 45')
ax.set_title('Dorje Teas Time to Second Purchase [SYNTHETIC DATA]')
ax.set_xlabel('Days from first to second order')
ax.set_ylabel('Customers')
ax.legend()
plt.tight_layout()
plt.show()
summary_t2

## Subscription Attach and Monthly Revenue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,5))
sns.barplot(data=cohorts, x='cohort_month', y='subscription_attach_m2_pct', ax=axes[0], color='#B279A2')
sns.barplot(data=cohorts, x='cohort_month', y='subscription_monthly_rev', ax=axes[1], color='#72B7B2')
axes[0].set_title('Subscription Attach Rate by Cohort')
axes[1].set_title('Subscription Monthly Revenue by Cohort')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Cohort month')
axes[0].set_ylabel('Attach rate (%)')
axes[1].set_ylabel('Revenue (Rs.)')
fig.suptitle('Dorje Teas Subscription Cohort View [SYNTHETIC DATA]')
plt.tight_layout()
plt.show()
cohorts[['cohort_month','subscription_attach_m2_pct','subscription_subscribers_m2','subscription_monthly_rev']]

## Key Findings

- The April 2025 cohort is the First Flush seasonal peak in this sample model and should be read separately from normal months.
- The October 2024 cohort reflects a Diwali/gifting pattern: higher M0 revenue can coexist with weaker repeat behavior.
- Time to second purchase is the operating input for Day 21, Day 30, Day 45, and Day 60 lifecycle flows.
- Subscription attach should be read after trust and repeat behavior, not as a first-visit acquisition tactic.

## Operator Decisions This Analysis Supports

- Build separate First Flush Darjeeling retention logic for seasonal buyers.
- Do not judge Gift Buyer cohorts only by M0 AOV; track self-purchase conversion.
- Trigger product-specific replenishment flows when time-to-second-purchase drifts.
- Use LTV:CAC by cohort before increasing acquisition spend.

## What Real Dorje Data Would Improve

- Actual customer order history would produce real second-purchase timing by SKU.
- Actual subscription app data would separate Tea Club Darjeeling Black Monthly and Tea Club Darjeeling Green Monthly retention.
- Actual acquisition source on each cohort would improve LTV:CAC confidence.